In [1]:
import pandas as pd
import warnings
from matplotlib import MatplotlibDeprecationWarning
warnings.filterwarnings("ignore", category=MatplotlibDeprecationWarning)

import sys
from transformers import DistilBertTokenizer
sys.path.append("../../../splade_sans/learned-sparse-retrieval-1.0.0")
from lsr.transformer import LSR
from IPython.display import display, HTML
from matplotlib.colors import Normalize, to_hex
from matplotlib import cm
import numpy as np
import pyterrier as pt
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
if not pt.started():
    pt.init()

/tmp/ipykernel_221/3079255974.py:17: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_221/3079255974.py:18: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [2]:
ohsumed_docs = pd.read_pickle("/nfs/primary/sas_reranker/ohsumed_docs_w_t5base_sensitivity.pkl")
queries = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/queries.pkl")
qrels = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/qrels.pkl")

In [3]:
distil_model_path = "/nfs/primary/SPLADE/splade_class/lsr_package/outputs/hn_t5_retrain_scratch/model"
distil_model = LSR(distil_model_path)

relevance_model_path = "/nfs/primary/SPLADE/splade_class/lsr_package/outputs/splade_ohsumed_multiple_negative/model"
relevance_model = LSR(relevance_model_path)

In [4]:
relevance_reps = relevance_model.encode_docs(ohsumed_docs.text.tolist(), out_fmt = "np")
sensitivity_reps = distil_model.encode_docs(ohsumed_docs.text.tolist(), out_fmt = "np")

relevance_reps.max(), sensitivity_reps.max()

(2.1904747, 3.3253422)

In [5]:
relevance_reps.mean(), sensitivity_reps.mean()

(0.0009662466, 0.0019836319)

In [6]:
scaler = StandardScaler()
combined = np.vstack([relevance_reps, sensitivity_reps])
scaler.fit(combined)

StandardScaler()

In [153]:
def get_colored_tokens_html(tokens, scores, colour="Greens"):
    """
    Returns an HTML string with tokens color-coded by importance.
    """
    # Normalize scores between 0 and 1

    if colour == "green":
        cmap = cm.get_cmap("Greens")
    elif colour == "red":
        cmap = cm.get_cmap("Reds")
        
    colored_tokens = []

    for token, score in zip(tokens, scores):
        color = to_hex(cmap(score))
        span = f'<span style="background-color:{color}; padding:2px; border-radius:4px;">{token}</span>'
        colored_tokens.append(span)

    return " ".join(colored_tokens)


def get_colored_tokens_latex(tokens, scores, colour):

    if colour == "green":
        cmap = cm.get_cmap("Greens")
    elif colour == "red":
        cmap = cm.get_cmap("Reds")
        
    colored_tokens = []
    for token, score in zip(tokens, scores):
        rgba = cmap(score)[:3]  # Get R, G, B tuple
        r, g, b = [int(255 * c) for c in rgba]
        const = 100
        color = f"[RGB]{{{min(r + const, 255)},{min(g + const, 255)},{min(b + const, 255)}}}"

        box = f"\\colorbox{color}{{{token}}}"
        colored_tokens.append(box.replace("#", "\#"))

    return " ".join(colored_tokens)

In [154]:
def get_tokens_and_intensities(query, document, model):
    q_rep = model.encode_queries([query], out_fmt = "np")[0]
    d_rep = model.encode_docs([document], out_fmt = "np")[0]
    d_rep = d_rep.reshape(1, -1)
    d_rep = scaler.transform(d_rep)
    d_rep = d_rep.flatten()
    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
    # Tokenize using the model's tokenizer
    q_tokens = tokenizer.tokenize(query)
    d_tokens = tokenizer.tokenize(document)

    q_token_ids = tokenizer.convert_tokens_to_ids(q_tokens)
    d_token_ids = tokenizer.convert_tokens_to_ids(d_tokens)

    # Collect intensity values from representation vector
    q_token_values = [q_rep[token_id] for token_id in q_token_ids]
    d_token_values = [d_rep[token_id] for token_id in d_token_ids]

    
    return (q_tokens, q_token_values), (d_tokens, d_token_values)

In [155]:
def display_colours(text, model, colour = "green"):
    out = get_tokens_and_intensities(text, "paris is the main airport", model)[0]
    html = get_colored_tokens_html(out[0], out[1], colour)
    display(HTML(html))
    print(get_colored_tokens_latex(out[0], out[1], colour))
    

# for i in range(50):
#     display_colours(ohsumed_docs[ohsumed_docs["sensitivity"] == 0].head(100).text.iloc[i], relevance_model)
#     print()
#     display_colours(ohsumed_docs[ohsumed_docs["sensitivity"] == 0].head(100).text.iloc[i], distil_model)
    

#     print("-" * 1000)
#     print()

In [156]:
sensitive_doc = ohsumed_docs[ohsumed_docs.apply(lambda row: row.astype(str).str.contains('How often does maternal').any(), axis=1)].text.tolist()[0]
non_sensitive_doc = ohsumed_docs[ohsumed_docs.apply(lambda row: row.astype(str).str.contains('disorder reported in the past was associated with little or no bleeding').any(), axis=1)].text.tolist()[0]

In [157]:
display_colours(sensitive_doc, relevance_model, "red")
print()
display_colours(sensitive_doc, distil_model, "red")
print()
print()
display_colours(non_sensitive_doc, relevance_model, "green")
print()
display_colours(non_sensitive_doc, distil_model, "green")

\colorbox[RGB]{255,255,255}{how} \colorbox[RGB]{255,234,202}{often} \colorbox[RGB]{255,255,255}{does} \colorbox[RGB]{203,100,112}{maternal} \colorbox[RGB]{203,100,112}{pre} \colorbox[RGB]{255,255,255}{\#\#ec} \colorbox[RGB]{255,153,141}{\#\#lam} \colorbox[RGB]{255,122,128}{\#\#ps} \colorbox[RGB]{255,255,255}{\#\#ia} \colorbox[RGB]{255,255,232}{-} \colorbox[RGB]{203,100,112}{ec} \colorbox[RGB]{255,153,141}{\#\#lam} \colorbox[RGB]{255,122,128}{\#\#ps} \colorbox[RGB]{255,255,255}{\#\#ia} \colorbox[RGB]{203,100,112}{inc} \colorbox[RGB]{203,100,112}{\#\#ite} \colorbox[RGB]{203,100,112}{th} \colorbox[RGB]{255,255,255}{\#\#rom} \colorbox[RGB]{255,255,255}{\#\#bo} \colorbox[RGB]{255,231,199}{\#\#cy} \colorbox[RGB]{255,255,255}{\#\#top} \colorbox[RGB]{203,100,112}{\#\#enia} \colorbox[RGB]{255,255,255}{in} \colorbox[RGB]{255,121,126}{the} \colorbox[RGB]{203,100,112}{fe} \colorbox[RGB]{203,100,112}{\#\#tus} \colorbox[RGB]{255,255,255}{?} \colorbox[RGB]{255,255,255}{[} \colorbox[RGB]{255,255,255}{

\colorbox[RGB]{255,255,255}{how} \colorbox[RGB]{255,255,255}{often} \colorbox[RGB]{255,255,255}{does} \colorbox[RGB]{255,113,120}{maternal} \colorbox[RGB]{255,226,194}{pre} \colorbox[RGB]{255,255,255}{\#\#ec} \colorbox[RGB]{255,255,255}{\#\#lam} \colorbox[RGB]{255,255,255}{\#\#ps} \colorbox[RGB]{255,255,255}{\#\#ia} \colorbox[RGB]{255,229,197}{-} \colorbox[RGB]{255,255,235}{ec} \colorbox[RGB]{255,255,255}{\#\#lam} \colorbox[RGB]{255,255,255}{\#\#ps} \colorbox[RGB]{255,255,255}{\#\#ia} \colorbox[RGB]{238,108,117}{inc} \colorbox[RGB]{203,100,112}{\#\#ite} \colorbox[RGB]{255,255,255}{th} \colorbox[RGB]{255,255,255}{\#\#rom} \colorbox[RGB]{255,255,255}{\#\#bo} \colorbox[RGB]{255,255,255}{\#\#cy} \colorbox[RGB]{255,255,255}{\#\#top} \colorbox[RGB]{203,100,112}{\#\#enia} \colorbox[RGB]{255,189,163}{in} \colorbox[RGB]{203,100,112}{the} \colorbox[RGB]{255,255,255}{fe} \colorbox[RGB]{255,255,255}{\#\#tus} \colorbox[RGB]{255,255,255}{?} \colorbox[RGB]{255,255,255}{[} \colorbox[RGB]{255,255,255}{

\colorbox[RGB]{100,168,127}{pulmonary} \colorbox[RGB]{100,168,127}{em} \colorbox[RGB]{121,226,158}{\#\#bol} \colorbox[RGB]{159,255,188}{\#\#us} \colorbox[RGB]{210,255,215}{-} \colorbox[RGB]{100,168,127}{induced} \colorbox[RGB]{100,168,127}{di} \colorbox[RGB]{229,255,229}{\#\#sse} \colorbox[RGB]{100,200,140}{\#\#minated} \colorbox[RGB]{100,202,141}{intra} \colorbox[RGB]{255,255,255}{\#\#vas} \colorbox[RGB]{112,219,152}{\#\#cular} \colorbox[RGB]{100,168,127}{coa} \colorbox[RGB]{100,207,143}{\#\#gul} \colorbox[RGB]{100,179,131}{\#\#ation} \colorbox[RGB]{255,255,255}{.} \colorbox[RGB]{100,168,127}{pulmonary} \colorbox[RGB]{100,168,127}{em} \colorbox[RGB]{121,226,158}{\#\#bol} \colorbox[RGB]{159,255,188}{\#\#us} \colorbox[RGB]{131,236,166}{as} \colorbox[RGB]{255,255,255}{a} \colorbox[RGB]{100,168,127}{cause} \colorbox[RGB]{255,255,255}{of} \colorbox[RGB]{100,168,127}{di} \colorbox[RGB]{229,255,229}{\#\#sse} \colorbox[RGB]{100,200,140}{\#\#minated} \colorbox[RGB]{100,202,141}{intra} \colorbo

\colorbox[RGB]{100,168,127}{pulmonary} \colorbox[RGB]{100,168,127}{em} \colorbox[RGB]{255,255,255}{\#\#bol} \colorbox[RGB]{143,248,175}{\#\#us} \colorbox[RGB]{236,255,234}{-} \colorbox[RGB]{255,255,255}{induced} \colorbox[RGB]{100,197,139}{di} \colorbox[RGB]{255,255,255}{\#\#sse} \colorbox[RGB]{255,255,255}{\#\#minated} \colorbox[RGB]{100,177,130}{intra} \colorbox[RGB]{255,255,255}{\#\#vas} \colorbox[RGB]{105,213,148}{\#\#cular} \colorbox[RGB]{100,168,127}{coa} \colorbox[RGB]{255,255,255}{\#\#gul} \colorbox[RGB]{255,255,255}{\#\#ation} \colorbox[RGB]{255,255,255}{.} \colorbox[RGB]{100,168,127}{pulmonary} \colorbox[RGB]{100,168,127}{em} \colorbox[RGB]{255,255,255}{\#\#bol} \colorbox[RGB]{143,248,175}{\#\#us} \colorbox[RGB]{242,255,239}{as} \colorbox[RGB]{255,255,255}{a} \colorbox[RGB]{255,255,255}{cause} \colorbox[RGB]{255,255,255}{of} \colorbox[RGB]{100,197,139}{di} \colorbox[RGB]{255,255,255}{\#\#sse} \colorbox[RGB]{255,255,255}{\#\#minated} \colorbox[RGB]{100,177,130}{intra} \colorbo

Query with the most relevant sensitive documents
	- Document that has the biggest change up
	- Document that has the biggest change down
Query with the least relevant sensitive documents
	- Document that has the biggest change up
	- Document that has the biggest change downe down

In [12]:
# Query with most relevant sensitive documents (53)
max_val = 0

for _, query in pd.merge(qrels, ohsumed_docs).groupby("qid"):
    if query[query["label"] >= 1].sensitivity.sum() > max_val:
        max_val = query[query["label"] >= 1].sensitivity.sum()
        max_qid = query.iloc[0]["qid"]

max_qid

# Query with least relevant sensitive documents (10)
min_val = 10000000000000000000

for _, query in pd.merge(qrels, ohsumed_docs).groupby("qid"):
    if query[query["label"] >= 1].sensitivity.sum() < min_val:
        min_val = query[query["label"] >= 1].sensitivity.sum()
        min_qid = query.iloc[0]["qid"]

min_qid

'10'

In [13]:
relevance = pt.io.read_results("../runs/splade_relevance % 100")
distill = pt.io.read_results("../runs/splade_distil % 100")
combined = pd.merge(relevance, distill, on = ["docno", "qid"])
combined["rank_change"] = combined["rank_x"] - combined["rank_y"] # positive = down from rel to sens. negative = up from rel to sens.
sensitivities = ohsumed_docs[["docno", "sensitivity"]]
combined = pd.merge(combined, sensitivities, on = "docno")
combined = pd.merge(combined, qrels, on = ["docno", "qid"], how = "left")

In [14]:
combined[combined["qid"] == max_qid]

,qid,docno,rank_x,score_x,name_x,rank_y,score_y,name_y,rank_change,sensitivity,label
1531,53,267139,4,105766.0,pyterrier,21,73909.0,pyterrier,-17,0,NaN
1532,53,56730,13,97990.0,pyterrier,95,66288.0,pyterrier,-82,1,2.0
1533,53,314399,50,88117.0,pyterrier,4,80258.0,pyterrier,46,1,2.0
1534,53,279708,60,84852.0,pyterrier,94,66337.0,pyterrier,-34,1,2.0
1535,53,153756,95,77103.0,pyterrier,32,72372.0,pyterrier,63,1,NaN


In [15]:
# Biggest increase rel to sens (153756) rank 95 -> 32 (qid 53)
# Biggest decrease from rel to sens (56730) rank 13 -> 95 (qid 53)
print(f"Query with the most relevant sensitive documnents. ('{queries[queries['qid'] == max_qid]['query'].iloc[0]}')\n")
print("Document with the largest increase in rank when ranked using relevance vs sensitivity SPLADE (rank 95 -> 32).")
print("Relevance SPLADE:")
display_colours(ohsumed_docs[ohsumed_docs["docno"] == "153756"].text.iloc[0], relevance_model, "red")
print("Sensitivity SPLADE:")
display_colours(ohsumed_docs[ohsumed_docs["docno"] == "153756"].text.iloc[0], distil_model, "red")

print()
print("Document with the largest decrease in rank when ranked using relevance vs sensitivity SPLADE (rank 13 -> 95):")
print("Relevance SPLADE:")
display_colours(ohsumed_docs[ohsumed_docs["docno"] == "56730"].text.iloc[0], relevance_model, "red")
print("Sensitivity SPLADE:")
display_colours(ohsumed_docs[ohsumed_docs["docno"] == "56730"].text.iloc[0], distil_model, "red")


# Biggest increase from rel to sens (106899) rank 97 -> 0
# Biggest decrease from rel to sense (347443) rank 4 -> 41
print(f"Query with the lease relevant sensitive documnents. ('{queries[queries['qid'] == min_qid]['query'].iloc[0]}')\n")
print("Document with the largest increase in rank when ranked using relevance vs sensitivity SPLADE (rank 97 -> 0):")
print("Relevance SPLADE:")
display_colours(ohsumed_docs[ohsumed_docs["docno"] == "106899"].text.iloc[0], relevance_model, "green")
print("Sensitivity SPLADE:")
display_colours(ohsumed_docs[ohsumed_docs["docno"] == "106899"].text.iloc[0], distil_model, "green")

print()
print("Document with the largest decrease in rank when ranked using relevance vs sensitivity SPLADE (rank 4 -> 41):")
print("Relevance SPLADE:")
display_colours(ohsumed_docs[ohsumed_docs["docno"] == "347443"].text.iloc[0], relevance_model, "green")
print("Sensitivity SPLADE:")

display_colours(ohsumed_docs[ohsumed_docs["docno"] == "347443"].text.iloc[0], distil_model, "green")

Query with the most relevant sensitive documnents. ('lupus nephritis, diagnosis and management')

Document with the largest increase in rank when ranked using relevance vs sensitivity SPLADE (rank 95 -> 32).
Relevance SPLADE:


\colorbox[RGB]{103,0,12}{lu} \colorbox[RGB]{103,0,12}{##pus} \colorbox[RGB]{103,0,12}{anti} \colorbox[RGB]{240,65,48}{##co} \colorbox[RGB]{255,245,240}{##ag} \colorbox[RGB]{103,0,12}{##ula} \colorbox[RGB]{208,29,31}{##nt} \colorbox[RGB]{219,39,35}{and} \colorbox[RGB]{106,0,13}{ce} \colorbox[RGB]{255,245,240}{##re} \colorbox[RGB]{239,61,45}{##bro} \colorbox[RGB]{244,81,58}{##vas} \colorbox[RGB]{103,0,12}{##cular} \colorbox[RGB]{103,0,12}{accident} \colorbox[RGB]{254,231,221}{in} \colorbox[RGB]{255,245,240}{a} \colorbox[RGB]{212,33,32}{patient} \colorbox[RGB]{255,245,240}{with} \colorbox[RGB]{103,0,12}{ne} \colorbox[RGB]{252,155,125}{##uro} \colorbox[RGB]{227,47,39}{##fi} \colorbox[RGB]{239,61,45}{##bro} \colorbox[RGB]{138,8,17}{##mat} \colorbox[RGB]{103,0,12}{##osis} \colorbox[RGB]{252,172,144}{.} \colorbox[RGB]{255,245,240}{this} \colorbox[RGB]{255,245,240}{report} \colorbox[RGB]{255,245,240}{describes} \colorbox[RGB]{238,58,43}{the} \colorbox[RGB]{206,27,30}{case} \colorbox[RGB]{255,2

\colorbox[RGB]{103,0,12}{lu} \colorbox[RGB]{255,245,240}{##pus} \colorbox[RGB]{103,0,12}{anti} \colorbox[RGB]{147,10,18}{##co} \colorbox[RGB]{255,245,240}{##ag} \colorbox[RGB]{252,151,120}{##ula} \colorbox[RGB]{255,245,240}{##nt} \colorbox[RGB]{147,10,18}{and} \colorbox[RGB]{103,0,12}{ce} \colorbox[RGB]{255,245,240}{##re} \colorbox[RGB]{255,245,240}{##bro} \colorbox[RGB]{255,245,240}{##vas} \colorbox[RGB]{251,117,85}{##cular} \colorbox[RGB]{103,0,12}{accident} \colorbox[RGB]{231,51,40}{in} \colorbox[RGB]{251,132,100}{a} \colorbox[RGB]{255,245,240}{patient} \colorbox[RGB]{253,213,195}{with} \colorbox[RGB]{255,245,240}{ne} \colorbox[RGB]{103,0,12}{##uro} \colorbox[RGB]{103,0,12}{##fi} \colorbox[RGB]{255,245,240}{##bro} \colorbox[RGB]{254,226,213}{##mat} \colorbox[RGB]{251,129,97}{##osis} \colorbox[RGB]{251,139,107}{.} \colorbox[RGB]{255,245,240}{this} \colorbox[RGB]{255,245,240}{report} \colorbox[RGB]{255,245,240}{describes} \colorbox[RGB]{174,17,23}{the} \colorbox[RGB]{255,245,240}{case

\colorbox[RGB]{103,0,12}{lu} \colorbox[RGB]{103,0,12}{##pus} \colorbox[RGB]{103,0,12}{ne} \colorbox[RGB]{252,204,183}{##ph} \colorbox[RGB]{103,0,12}{##rit} \colorbox[RGB]{175,17,23}{##is} \colorbox[RGB]{255,245,240}{:} \colorbox[RGB]{103,0,12}{clinical} \colorbox[RGB]{253,208,189}{and} \colorbox[RGB]{185,19,25}{path} \colorbox[RGB]{124,5,15}{##ological} \colorbox[RGB]{103,0,12}{correlation} \colorbox[RGB]{252,146,114}{.} \colorbox[RGB]{245,86,61}{the} \colorbox[RGB]{103,0,12}{clinical} \colorbox[RGB]{136,8,17}{course} \colorbox[RGB]{255,245,240}{of} \colorbox[RGB]{255,245,240}{135} \colorbox[RGB]{252,175,147}{patients} \colorbox[RGB]{255,245,240}{with} \colorbox[RGB]{103,0,12}{lu} \colorbox[RGB]{103,0,12}{##pus} \colorbox[RGB]{103,0,12}{ne} \colorbox[RGB]{252,204,183}{##ph} \colorbox[RGB]{103,0,12}{##rit} \colorbox[RGB]{175,17,23}{##is} \colorbox[RGB]{255,245,240}{was} \colorbox[RGB]{254,241,235}{examined} \colorbox[RGB]{245,86,61}{long} \colorbox[RGB]{252,175,147}{-} \colorbox[RGB]{23

\colorbox[RGB]{103,0,12}{lu} \colorbox[RGB]{255,245,240}{##pus} \colorbox[RGB]{110,1,14}{ne} \colorbox[RGB]{255,245,240}{##ph} \colorbox[RGB]{255,245,240}{##rit} \colorbox[RGB]{103,0,12}{##is} \colorbox[RGB]{255,245,240}{:} \colorbox[RGB]{247,93,66}{clinical} \colorbox[RGB]{181,18,24}{and} \colorbox[RGB]{255,245,240}{path} \colorbox[RGB]{255,245,240}{##ological} \colorbox[RGB]{252,199,177}{correlation} \colorbox[RGB]{225,46,38}{.} \colorbox[RGB]{122,4,15}{the} \colorbox[RGB]{247,93,66}{clinical} \colorbox[RGB]{255,245,240}{course} \colorbox[RGB]{255,245,240}{of} \colorbox[RGB]{255,245,240}{135} \colorbox[RGB]{255,245,240}{patients} \colorbox[RGB]{255,245,240}{with} \colorbox[RGB]{103,0,12}{lu} \colorbox[RGB]{255,245,240}{##pus} \colorbox[RGB]{110,1,14}{ne} \colorbox[RGB]{255,245,240}{##ph} \colorbox[RGB]{255,245,240}{##rit} \colorbox[RGB]{103,0,12}{##is} \colorbox[RGB]{255,245,240}{was} \colorbox[RGB]{255,245,240}{examined} \colorbox[RGB]{134,7,17}{long} \colorbox[RGB]{251,115,83}{-} \

\colorbox[RGB]{0,68,27}{gall} \colorbox[RGB]{0,68,27}{##ium} \colorbox[RGB]{107,191,113}{-} \colorbox[RGB]{0,68,27}{67} \colorbox[RGB]{0,68,27}{up} \colorbox[RGB]{0,68,27}{##take} \colorbox[RGB]{34,138,68}{by} \colorbox[RGB]{230,245,225}{a} \colorbox[RGB]{0,68,27}{benign} \colorbox[RGB]{37,141,70}{ad} \colorbox[RGB]{56,162,86}{##ren} \colorbox[RGB]{195,231,188}{##oco} \colorbox[RGB]{0,97,39}{##rti} \colorbox[RGB]{13,120,53}{##cal} \colorbox[RGB]{0,68,27}{aden} \colorbox[RGB]{0,68,27}{##oma} \colorbox[RGB]{182,225,175}{.} \colorbox[RGB]{230,245,225}{a} \colorbox[RGB]{247,252,245}{55} \colorbox[RGB]{107,191,113}{-} \colorbox[RGB]{0,68,27}{y} \colorbox[RGB]{247,252,245}{##r} \colorbox[RGB]{107,191,113}{-} \colorbox[RGB]{247,252,245}{old} \colorbox[RGB]{247,252,245}{man} \colorbox[RGB]{247,252,245}{presented} \colorbox[RGB]{247,252,245}{with} \colorbox[RGB]{247,252,245}{an} \colorbox[RGB]{50,155,81}{at} \colorbox[RGB]{230,245,225}{##yp} \colorbox[RGB]{209,237,202}{##ical} \colorbox[RGB]{11

\colorbox[RGB]{0,68,27}{gall} \colorbox[RGB]{0,68,27}{##ium} \colorbox[RGB]{89,183,105}{-} \colorbox[RGB]{35,139,69}{67} \colorbox[RGB]{0,68,27}{up} \colorbox[RGB]{75,176,97}{##take} \colorbox[RGB]{185,227,178}{by} \colorbox[RGB]{146,210,142}{a} \colorbox[RGB]{189,228,182}{benign} \colorbox[RGB]{0,68,27}{ad} \colorbox[RGB]{247,252,245}{##ren} \colorbox[RGB]{234,247,230}{##oco} \colorbox[RGB]{247,252,245}{##rti} \colorbox[RGB]{149,211,145}{##cal} \colorbox[RGB]{0,68,27}{aden} \colorbox[RGB]{21,126,58}{##oma} \colorbox[RGB]{146,210,142}{.} \colorbox[RGB]{146,210,142}{a} \colorbox[RGB]{247,252,245}{55} \colorbox[RGB]{89,183,105}{-} \colorbox[RGB]{247,252,245}{y} \colorbox[RGB]{247,252,245}{##r} \colorbox[RGB]{89,183,105}{-} \colorbox[RGB]{196,231,189}{old} \colorbox[RGB]{126,200,126}{man} \colorbox[RGB]{247,252,245}{presented} \colorbox[RGB]{242,250,239}{with} \colorbox[RGB]{234,247,230}{an} \colorbox[RGB]{168,220,162}{at} \colorbox[RGB]{247,252,245}{##yp} \colorbox[RGB]{247,252,245}{##ic

\colorbox[RGB]{247,252,245}{a} \colorbox[RGB]{0,109,44}{random} \colorbox[RGB]{247,252,245}{##ized} \colorbox[RGB]{0,68,27}{double} \colorbox[RGB]{135,204,133}{-} \colorbox[RGB]{0,68,27}{blind} \colorbox[RGB]{41,146,74}{study} \colorbox[RGB]{247,252,245}{of} \colorbox[RGB]{0,68,27}{gall} \colorbox[RGB]{0,93,37}{##ium} \colorbox[RGB]{0,68,27}{nitrate} \colorbox[RGB]{29,134,65}{compared} \colorbox[RGB]{247,252,245}{with} \colorbox[RGB]{49,154,80}{et} \colorbox[RGB]{247,252,245}{##id} \colorbox[RGB]{64,170,92}{##rona} \colorbox[RGB]{7,115,49}{##te} \colorbox[RGB]{247,252,245}{for} \colorbox[RGB]{55,161,85}{acute} \colorbox[RGB]{0,68,27}{control} \colorbox[RGB]{247,252,245}{of} \colorbox[RGB]{0,68,27}{cancer} \colorbox[RGB]{135,204,133}{-} \colorbox[RGB]{0,92,37}{related} \colorbox[RGB]{25,130,62}{hyper} \colorbox[RGB]{0,68,27}{##cal} \colorbox[RGB]{46,151,78}{##ce} \colorbox[RGB]{0,68,27}{##mia} \colorbox[RGB]{125,200,125}{.} \colorbox[RGB]{25,130,62}{hyper} \colorbox[RGB]{0,68,27}{##cal}

\colorbox[RGB]{239,249,236}{a} \colorbox[RGB]{247,252,245}{random} \colorbox[RGB]{247,252,245}{##ized} \colorbox[RGB]{247,252,245}{double} \colorbox[RGB]{153,213,148}{-} \colorbox[RGB]{247,252,245}{blind} \colorbox[RGB]{247,252,245}{study} \colorbox[RGB]{244,250,241}{of} \colorbox[RGB]{167,219,161}{gall} \colorbox[RGB]{0,68,27}{##ium} \colorbox[RGB]{0,68,27}{nitrate} \colorbox[RGB]{247,252,245}{compared} \colorbox[RGB]{247,252,245}{with} \colorbox[RGB]{0,68,27}{et} \colorbox[RGB]{223,242,217}{##id} \colorbox[RGB]{0,68,27}{##rona} \colorbox[RGB]{22,127,59}{##te} \colorbox[RGB]{245,251,243}{for} \colorbox[RGB]{247,252,245}{acute} \colorbox[RGB]{0,68,27}{control} \colorbox[RGB]{244,250,241}{of} \colorbox[RGB]{97,186,108}{cancer} \colorbox[RGB]{153,213,148}{-} \colorbox[RGB]{235,247,231}{related} \colorbox[RGB]{161,217,155}{hyper} \colorbox[RGB]{0,68,27}{##cal} \colorbox[RGB]{63,169,91}{##ce} \colorbox[RGB]{65,171,93}{##mia} \colorbox[RGB]{126,200,126}{.} \colorbox[RGB]{161,217,155}{hyper}